# Congestion Rent: 2026's Most Widely Congested Moment

When was the grid most *widely* constrained in 2026 &ndash; the single moment with the most borders congested at once?

**Before you run this:** sign up at [voltpath.ai/dashboard](https://voltpath.ai/dashboard) for a free personal API key (copy it any time from [voltpath.ai/settings](https://voltpath.ai/settings)), then copy `.env.example` to `.env` and paste it in as `VOLTPATH_API_KEY`.

**Data source:** [ENTSO-E Transparency Platform](https://transparency.entsoe.eu), via the [Voltpath API](https://voltpath.ai) (docs at [docs.voltpath.ai](https://docs.voltpath.ai)).

In [ ]:
# common.py is shared across every article and changes often; without this a
# kernel that already imported it keeps serving the stale module.
%load_ext autoreload
%autoreload 2

import sys
from datetime import UTC, datetime
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from common import (
    BALTIC_ZONES,
    ZONE_REGIONS,
    congestion_rent_table,
    fetch_flows_range,
    hourly_border_counts,
    zone_border_breakdown,
    zone_price_summary,
)

## Case study: 2026's most widely congested moments

We scan every Market Time Unit (MTU) across 2026, count how many borders are flagged `Congestion`, then take the timestamps with the highest counts.

In [ ]:
end = min(datetime.now(UTC), datetime(2027, 1, 1, tzinfo=UTC))

# We'll use VoltPath APIs to fetch flows data
flows_2026 = fetch_flows_range("2026-01-01", end.strftime("%Y-%m-%d"))

counts_2026 = hourly_border_counts(flows_2026, "Congestion")
total_borders = flows_2026["border"].nunique()

max_borders = counts_2026["n_borders"].max()
tied = counts_2026[counts_2026["n_borders"] == max_borders].sort_values("datetime")
print(
    (
        f"Peak: {max_borders} of {total_borders} borders simultaneously congested, "
        f"{len(tied)} timestamp(s) tied at that max:"
    )
)
tied

Let's look at each snapshot in more detail.

## Price spike decouples Baltics + Finland from Europe

The [VoltPath dashboard set to 2026-02-03](https://voltpath.ai/dashboard?viewMode=flows-prices&dateFrom=2026-02-03&dateTo=2026-02-04) &ndash; with the playback control dragged to 13:15 UTC &ndash; shows the first peak. Congested flows are marked with red arrows.

<img src="media/congestion-peak-2026-02-03-dashboard.png" alt="VoltPath dashboard showing peak simultaneous congestion on 2026-02-03" width="800">

Each highly congested snapshot will tell us a different story, and this one is best seen by looking at prices across the entire day (which VoltPath supports for any date-range selection). Upon playing back prices, it's hard not to notice that FI, EE, LV and LT exhibited significantly higher prices than the rest of Europe throughout the day on Feb 3.

<img src="media/congestion-2026-02-03-day-prices.gif" alt="VoltPath dashboard day-long price playback for 2026-02-03" width="800">

Putting that into numbers, the Baltic bidding zones (BZs) and Finland produced prices 3-4x higher than the average prices elsewhere in Europe:

In [ ]:
BALTIC_FI_ZONES = BALTIC_ZONES | {"FI"}
peak_20260203 = tied["datetime"].iloc[0]
zone_price_summary(flows_2026, BALTIC_FI_ZONES, "Baltic + FI", peak_20260203)

Three factors were at play behind this isolated price spike:

1. **Weather** &ndash; a strong cold snap drove heating demand up, while Finland experienced some of the lowest wind speeds in three decades. [Seinäjoki University of Applied Sciences reported](https://lehti.seamk.fi/verkkolehti/tuulen-nopeuden-poikkeuksellisuus-tammi-helmikuussa-2026/) that the average wind speed at coastal stations fell below 7 m/s.
   
2. **Supply-side squeeze** &ndash; [Estonia's Auvere power plant was down](https://news.err.ee/1609911433/auvere-power-plant-to-take-weeks-to-repair) for weeks due to a crack in the masonry lining around one of the plant's four heat exchangers, taking out ~260-270 MW of capacity from generation. Simultaneously, Elering, Estonia's Transmission System Operator, cut the Estonia->Latvia transfer capacity [from 1447 MW to 922 MW](https://www.err.ee/1609893697/lati-suunaline-elektri-ulekandevoimsus-jaab-vaiksemaks-kui-tootjad-ootasid) in January 2026, congesting the region's import-constrained borders even further.

3. **Market design** &ndash; How come Poland, where winters are equally harsh, and where cheap Nordic electricity is usually imported in through LT, was able to avoid the extreme prices we saw for Baltic + FI? Poland, at the edge of the Core region, is part of a *meshed* network with 6 cross-border connections. It is interconnected, in contrast to the *radial*, sequential network of the Baltic region with 2-3 borders for each Baltic state. 

On the last point, Poland not only escaped the pricy side of the regional split but, in fact, it arbitraged the cheaper German electricity onward, buying at €92.02/MWh and reselling part of it to Lithuania at €484.42 (note that these are day-ahead prices) &ndash; just over 4x the Polish price.

In [ ]:
zone_border_breakdown(flows_2026, "PL", peak_20260203)

All in all, extreme winter conditions, and weakened and 'cornered' generation and transmission capacity congested northern BZs, added to which the generally congested core region (Central Europe) produced the most congested moment of 2026's winter. Next, we'll look at how the tables have turned 4 months later.

## Core Europe spikes while the Nordics stay flush

The [VoltPath dashboard for 2026-06-20](https://voltpath.ai/dashboard?viewMode=flows-prices&dateFrom=2026-06-20&dateTo=2026-06-21) &ndash; with the playback control dragged to 20:15 UTC &ndash; shows the second tied peak, this time during peak consumption hours in the evening.

<img src="media/congestion-peak-2026-06-20-dashboard.png" alt="VoltPath dashboard showing peak simultaneous congestion on 2026-06-20" width="800">

Replaying the day again, we see some interesting trends.

<img src="media/congestion-2026-06-20-day-prices.gif" alt="VoltPath dashboard day-long price playback for 2026-06-20" width="800">

For a large chunk of Jun 20, prices floated around zero. But as soon as the day enters the afternoon/evening peak hours, we see a split between north and south again, this time the Nordic and Baltic regions faring the weather significantly better.

This particular split shows that while capturing essentially free electricity at scale is an achievement Europe should be proud of, the Core region's heavy reliance on nuclear left it exposed once a heatwave constrained cooling-water-limited output.

The spread was not as large as in the case of Feb 2 in the Baltics, but the Core region (CE) exhibits clearly higher average prices:

In [ ]:
CORE_ZONES = {zone for zone, region in ZONE_REGIONS.items() if region == "Core"}
peak_20260620 = tied["datetime"].iloc[1]
zone_price_summary(flows_2026, CORE_ZONES, "Core", peak_20260620)

So, once again, two factors surface behind regional price splits, and consequently, continent-wide congestion:

1. **Weather** &ndash; June saw heatwave records shatter, pushing temperature predictions for July to [2-8°C above seasonal norms](https://gmk.center/en/infographic/electricity-prices-in-europe-rose-in-june-amid-a-heatwave/).
2. **Supply-side squeeze** &ndash; French nuclear plants [cut or shut output](https://oenergetice.cz/en/nuclear-power/heat-curbs-nuclear-output-frances-edf-shuts-down-more-units) due to river cooling-water temperature limits. [Reuters reported](https://www.reuters.com/business/energy/europes-heatwave-curbs-french-nuclear-plants-2026-06-24/) that in some instances output was reduced by 4.1 gigawatts, amounting to 7% of total power demand.

The Nordics and Baltics missed out on the fun thanks to being reliant on hydro, wind and thermal. 

But what happens when transmission is congested but no weather-related events or structural shifts are at play?

## Congestion ≠ Crisis

Lastly, the final highly-congested snapshot: [VoltPath dashboard for 2026-09-19](https://voltpath.ai/dashboard?viewMode=flows-prices&dateFrom=2026-09-19&dateTo=2026-09-20) with the playback control dragged to 12:45 UTC.

<img src="media/congestion-peak-2026-09-19-dashboard.png" alt="VoltPath dashboard showing peak simultaneous congestion on 2026-09-19" width="800">

When Europe is not experiencing frosty winters and dry summers, congestion can still occur, this time showing that the market clearing algorithm works very, very well.

<img src="media/congestion-2026-09-19-day-prices.gif" alt="VoltPath dashboard day-long price playback for 2026-09-19" width="800">

Following the morning peak hours, Europe is awash in cheap, clean electricity. The Nordic and Baltic BZs are profiting off of their wind, thermal and hydro plants, while many other European countries like Germany and Spain are benefiting from solar generation. **As a result, prices are not only near-zero, but are converging across the continent**.

Two regions deviate slightly: the Balkans and Italy. South-East Europe (SEE) has weaker interconnections, and less wind and solar buildout. In addition, many countries were still recovering from severe droughts: Romania's [Cernavodă](https://www.ansa.it/nuova_europa/en/news/sections/energy/2026/08/13/romania-begins-shutdown-of-nuclear-plant-due-dvia-low-danube_69797561-a288-4b4d-a5dc-d8eaac7cf9c3.html), Hungary's [Paks](https://balkangreenenergynews.com/heatwave-pushes-power-prices-past-eur-700-in-southeast-europe/) and Slovenia's [Krško](https://balkangreenenergynews.com/slovenian-nuclear-plant-krsko-to-cut-output-adding-pressure-to-southeast-europes-power-supply/) were all partially or fully shut down for river-cooling reasons.

While the difference is minimal, a trend is still noticeable in the average prices:



In [ ]:
BALKAN_ITALY_ZONES = {
    zone for zone, region in ZONE_REGIONS.items() if region in ("South-East Europe", "Greece-Italy")
}
peak_20260919 = tied["datetime"].iloc[2]
zone_price_summary(flows_2026, BALKAN_ITALY_ZONES, "Balkans + Italy", peak_20260919)

The last metric we'll calculate is known as **congestion rent** or congestion income. Congestion rent is the price gap x volume traded, as explained by ENTSO-E's [TSOs' proposal for the Use of Congestion Income &ndash; explanatory document](https://eepublicdownloads.entsoe.eu/clean-documents/cep/200703_Use_of_Congestion_Income_-_Explanatory_Document.pdf).

The reason for bringing in the concept now will become clear after calculating congestion rent for all three snapshots we examined in this analysis.

In [ ]:
congestion_rent_table(
    flows_2026,
    {"2026-02-03": peak_20260203, "2026-06-20": peak_20260620, "2026-09-19": peak_20260919},
)

In the hours surrounding the three snapshots, congestion rent was 6,546.3k,	3,742.2k, and 2,464.5k for Feb 3, Jun 20, and Sep 19, respectively. Placing these numbers into context will be the subject of another article, what we can see however is that Sep 19 produced one-third of congestion rent compared to Feb 3. My reading is that border count alone doesn't tell you how economically significant a congestion event is &ndash; Sep 19 congested just as many borders as Feb 3, but at a fraction of the cost, because most of Europe had already converged on near-zero prices.

## Final thoughts

The point to take away here is that congestion can be a sign of success as well as a sympton of structural weakness. Deciding it without context and reproducable data, however, is impossible. That is why we created [VoltPath](voltpath.ai) and if you agree, [we would love to hear from you](https://voltpath.ai/about-us).